In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib, numpy as np, pandas as pd, torch
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
from metrics import evaluate

BASE = "/content/drive/MyDrive/esci"
df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
df = df.merge(qs, on="query_id", how="left")
assert df["rule"].notna().all()

# leak checks (test only used here, then dropped)
ids = {p: set(df.loc[df.part == p, "query_id"]) for p in ["train", "val", "test"]}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
norm = lambda s: " ".join(str(s).lower().split())
assert not (set(df.loc[df.part == "train", "query"].map(norm)) & set(df.loc[df.part == "val", "query"].map(norm)))
train = df[df.part == "train"].reset_index(drop=True)
val   = df[df.part == "val"].reset_index(drop=True)
del df

# M3 category stand-in: Rule A only (Rule B category comes from labelled products = leak)
CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
for d in (train, val):
    d["category"] = np.where(d["rule"] == "A", d["scope_group"].map(CAT), None)
    d["q_raw"]         = d["query"].map(C._clean)
    d["q_raw_cat"]     = [C.query_to_text(q, category=c, use_rules=False) for q, c in zip(d["query"], d["category"])]
    d["q_focused"]     = [C.query_to_text(q) for q in d["query"]]
    d["q_focused_cat"] = [C.query_to_text(q, category=c) for q, c in zip(d["query"], d["category"])]
    prods = [C.esci_row_to_product(r) for r in d.to_dict("records")]
    for s in C.STYLES:
        d[f"text_{s}"] = [C.product_to_text(p, s) for p in prods]

QUERY_INPUTS = ["q_raw", "q_raw_cat", "q_focused", "q_focused_cat"]
print("product formats:", C.STYLES, "| same in trainer:", T.STYLES == C.STYLES)
print("train rows", len(train), "| val rows", len(val), "| GPU", torch.cuda.get_device_name(0))
print("val queries with category:", val.loc[val.category.notna(), "query_id"].nunique(), "of", val.query_id.nunique())
ex = val.drop_duplicates("query_id").sample(4, random_state=3)
print(ex[["query"] + QUERY_INPUTS].to_string())
print(val[["text_title", "text_key_value"]].head(2).to_string())

Mounted at /content/drive
product formats: ['title', 'title_brand', 'full', 'full_bullets', 'key_value'] | same in trainer: True
train rows 65852 | val rows 11741 | GPU Tesla T4
val queries with category: 495 of 656
                         query                     q_raw                                   q_raw_cat                 q_focused                               q_focused_cat
4488     galaxy watch active 2     galaxy watch active 2  [Mobile & Computing] galaxy watch active 2     galaxy watch active 2  [Mobile & Computing] galaxy watch active 2
736                a8g oled tv               a8g oled tv              [Home Electronics] a8g oled tv               a8g oled tv              [Home Electronics] a8g oled tv
1892                  aux cord                  aux cord                                    aux cord                  aux cord                                    aux cord
8364  overhead headphone wired  overhead headphone wired            [Audio] overhead headphone wired

In [3]:
MODELS = {"minilm_l6":  "cross-encoder/ms-marco-MiniLM-L-6-v2",
          "minilm_l12": "cross-encoder/ms-marco-MiniLM-L-12-v2",
          "distilbert": "distilbert-base-uncased",
          "bert_base":  "bert-base-uncased"}
LOG = "s7_grid_val_log.csv"            # new log, separate from the old Step 5 log

def run_model(short, accum=1):
    for qin in QUERY_INPUTS:
        cfg = {**T.CFG, "query_col": qin, "styles": C.STYLES, "accum": accum}
        print(f"\n===== {short}  |  {qin} =====")
        T.train_run(MODELS[short], f"{short}__{qin}", train, val, evaluate, BASE, cfg=cfg, log_name=LOG)

In [4]:
run_model("minilm_l6")


===== minilm_l6  |  q_raw =====
minilm_l6__q_raw: starting at epoch 1 from cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

epoch 1 done: loss 0.4761, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9246 0.9410 0.8488
 title_brand   0.9259 0.9411 0.8479
        full   0.9262 0.9436 0.8485
full_bullets   0.9284 0.9405 0.8500
   key_value   0.9269 0.9418 0.8494
  epoch 2 step 500/2058 loss 0.3441
  epoch 2 step 1000/2058 loss 0.3417
  epoch 2 step 1500/2058 loss 0.3381
  epoch 2 step 2000/2058 loss 0.3348


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.3353, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9259 0.9424 0.8466
 title_brand   0.9255 0.9424 0.8473
        full   0.9251 0.9387 0.8473
full_bullets   0.9270 0.9386 0.8494
   key_value   0.9261 0.9388 0.8470
  epoch 3 step 500/2058 loss 0.3040
  epoch 3 step 1000/2058 loss 0.3046
  epoch 3 step 1500/2058 loss 0.3068
  epoch 3 step 2000/2058 loss 0.3046


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.3046, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9259 0.9415 0.8454
 title_brand   0.9271 0.9431 0.8479
        full   0.9265 0.9423 0.8473
full_bullets   0.9273 0.9383 0.8500
   key_value   0.9269 0.9394 0.8482


In [5]:
from scipy.stats import wilcoxon
LOGDF = pd.read_csv(f"{BASE}/results/{LOG}")
runs = [f"minilm_l6__{q}" for q in QUERY_INPUTS]

# 1) best epoch per run = highest mean NDCG@10 over the 5 formats
summ = LOGDF[LOGDF.run.isin(runs)].groupby(["run", "epoch"])[["ndcg@10", "mrr", "p@5"]].mean().reset_index()
best = summ.loc[summ.groupby("run")["ndcg@10"].idxmax()].reset_index(drop=True)
print("best epoch per run:\n", best.round(4).to_string(index=False))

# 2) per-query scores (mean over the 5 formats) at each run's best epoch
def perq(run, ep):
    d = pd.read_csv(f"{BASE}/models/{run}/per_query_val_epoch_{ep}.csv")
    return d.groupby("query_id")[["ndcg@10", "mrr", "p@5"]].mean()
P = {r: perq(r, int(e)) for r, e in zip(best.run, best.epoch)}

vq = val.drop_duplicates("query_id").set_index("query_id")
changed   = vq.index[vq["q_focused"] != vq["q_raw"]]
unchanged = vq.index[vq["q_focused"] == vq["q_raw"]]
ruleA, ruleB = vq.index[vq["rule"] == "A"], vq.index[vq["rule"] == "B"]
print(f"\nval queries: changed by query_to_text = {len(changed)}, unchanged = {len(unchanged)}, "
      f"Rule A = {len(ruleA)}, Rule B = {len(ruleB)}")

rows = []
for r, d in P.items():
    rows.append({"run": r, "all": d["ndcg@10"].mean(), "rule_A": d.loc[ruleA, "ndcg@10"].mean(),
                 "rule_B": d.loc[ruleB, "ndcg@10"].mean(), "changed_q": d.loc[changed, "ndcg@10"].mean(),
                 "unchanged_q": d.loc[unchanged, "ndcg@10"].mean()})
print("\nNDCG@10 by query group:\n", pd.DataFrame(rows).round(4).to_string(index=False))

# 3) paired comparison (same 656 queries): which differences are real?
pairs = [("q_raw", "q_raw_cat"), ("q_focused", "q_focused_cat"), ("q_raw", "q_focused"),
         ("q_raw_cat", "q_focused_cat"), ("q_raw", "q_focused_cat")]
for a, b in pairs:
    x = P[f"minilm_l6__{a}"]["ndcg@10"]; y = P[f"minilm_l6__{b}"]["ndcg@10"].loc[x.index]
    d = y - x; nz = d[d != 0]
    p = wilcoxon(nz).pvalue if len(nz) > 10 else float("nan")
    print(f"{b:14s} vs {a:10s}: mean diff {d.mean():+.4f} | better {int((d>0).sum())} / worse {int((d<0).sum())} queries"
          f" | diff on UNCHANGED queries {d.loc[unchanged].mean():+.4f} | Wilcoxon p = {p:.3f}")

best epoch per run:
                      run  epoch  ndcg@10    mrr    p@5
    minilm_l6__q_focused      3   0.9280 0.9429 0.8461
minilm_l6__q_focused_cat      3   0.9267 0.9409 0.8478
        minilm_l6__q_raw      3   0.9277 0.9409 0.8470
    minilm_l6__q_raw_cat      3   0.9294 0.9449 0.8489

val queries: changed by query_to_text = 4, unchanged = 652, Rule A = 495, Rule B = 161

NDCG@10 by query group:
                      run    all  rule_A  rule_B  changed_q  unchanged_q
    minilm_l6__q_focused 0.9280  0.9277  0.9290     0.9852       0.9276
minilm_l6__q_focused_cat 0.9268  0.9268  0.9265     0.9851       0.9264
        minilm_l6__q_raw 0.9277  0.9270  0.9297     0.9971       0.9273
    minilm_l6__q_raw_cat 0.9294  0.9289  0.9310     0.9971       0.9290
q_raw_cat      vs q_raw     : mean diff +0.0017 | better 136 / worse 132 queries | diff on UNCHANGED queries +0.0017 | Wilcoxon p = 0.306
q_focused_cat  vs q_focused : mean diff -0.0012 | better 136 / worse 141 queries | diff on U

In [6]:
def run_model(short, seed=42, accum=1, inputs=QUERY_INPUTS):
    for qin in inputs:
        cfg = {**T.CFG, "query_col": qin, "styles": C.STYLES, "accum": accum, "seed": seed}
        name = f"{short}__{qin}" if seed == 42 else f"{short}__{qin}__s{seed}"
        print(f"\n===== {name} =====")
        T.train_run(MODELS[short], name, train, val, evaluate, BASE, cfg=cfg, log_name=LOG)

In [7]:
run_model("minilm_l6", seed=43)
run_model("minilm_l6", seed=44)


===== minilm_l6__q_raw__s43 =====
minilm_l6__q_raw__s43: starting at epoch 1 from cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  epoch 1 step 500/2058 loss 0.6965
  epoch 1 step 1000/2058 loss 0.5621
  epoch 1 step 1500/2058 loss 0.5058
  epoch 1 step 2000/2058 loss 0.4726


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 1 done: loss 0.4705, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9244 0.9397 0.8470
 title_brand   0.9238 0.9368 0.8470
        full   0.9246 0.9380 0.8473
full_bullets   0.9282 0.9455 0.8479
   key_value   0.9250 0.9382 0.8479
  epoch 2 step 500/2058 loss 0.3436
  epoch 2 step 1000/2058 loss 0.3425
  epoch 2 step 1500/2058 loss 0.3404
  epoch 2 step 2000/2058 loss 0.3363


...

epoch 1 done: loss 0.4797, 1.8 min
       style  ndcg@10    mrr    p@5
       title   0.9224 0.9401 0.8463
 title_brand   0.9236 0.9429 0.8454
        full   0.9229 0.9416 0.8454
full_bullets   0.9247 0.9385 0.8479
   key_value   0.9229 0.9400 0.8470
  epoch 2 step 500/2058 loss 0.3481
  epoch 2 step 1000/2058 loss 0.3413
  epoch 2 step 1500/2058 loss 0.3376
  epoch 2 step 2000/2058 loss 0.3342


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.3339, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9270 0.9413 0.8500
 title_brand   0.9289 0.9416 0.8497
        full   0.9287 0.9414 0.8506
full_bullets   0.9288 0.9412 0.8479
   key_value   0.9291 0.9417 0.8494
  epoch 3 step 500/2058 loss 0.3066
  epoch 3 step 1000/2058 loss 0.3081
  epoch 3 step 1500/2058 loss 0.3066
  epoch 3 step 2000/2058 loss 0.3043


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.3038, 1.8 min
       style  ndcg@10    mrr    p@5
       title   0.9262 0.9386 0.8482
 title_brand   0.9273 0.9409 0.8473
        full   0.9269 0.9403 0.8460
full_bullets   0.9272 0.9392 0.8457
   key_value   0.9281 0.9439 0.8473


In [8]:
LOGDF = pd.read_csv(f"{BASE}/results/{LOG}")
L6 = LOGDF[LOGDF.run.str.startswith("minilm_l6__")].copy()
L6["qin"]  = L6.run.str.extract(r"minilm_l6__(q_[a-z_]+?)(?:__s\d+)?$")[0]
L6["seed"] = L6.run.str.extract(r"__s(\d+)$")[0].fillna("42").astype(int)

# best epoch per run (mean over 5 formats), then mean ± std over the 3 seeds
per_run = L6.groupby(["qin", "seed", "epoch"])[["ndcg@10", "mrr", "p@5"]].mean().reset_index()
best = per_run.loc[per_run.groupby(["qin", "seed"])["ndcg@10"].idxmax()]
print("best epoch per run:\n", best.round(4).to_string(index=False))
agg = best.groupby("qin")[["ndcg@10", "mrr", "p@5"]].agg(["mean", "std"]).round(4)
print("\nmean ± std over 3 seeds:\n", agg.to_string())

# per-query scores averaged over the 3 seeds, then paired Wilcoxon between inputs
def perq_avg(qin):
    frames = []
    for _, r in best[best.qin == qin].iterrows():
        name = f"minilm_l6__{qin}" if r.seed == 42 else f"minilm_l6__{qin}__s{r.seed}"
        d = pd.read_csv(f"{BASE}/models/{name}/per_query_val_epoch_{int(r.epoch)}.csv")
        frames.append(d.groupby("query_id")["ndcg@10"].mean())
    return pd.concat(frames, axis=1).mean(axis=1)
PQ = {q: perq_avg(q) for q in QUERY_INPUTS}
for a, b in [("q_raw", "q_raw_cat"), ("q_raw", "q_focused"), ("q_raw", "q_focused_cat"), ("q_raw_cat", "q_focused_cat")]:
    d = (PQ[b] - PQ[a].loc[PQ[b].index]); nz = d[d != 0]
    print(f"{b:14s} vs {a:10s}: mean diff {d.mean():+.4f} | Wilcoxon p = {wilcoxon(nz).pvalue:.3f}")
best.to_csv(f"{BASE}/results/s7_L6_seeds_summary.csv", index=False)

best epoch per run:
           qin  seed  epoch  ndcg@10    mrr    p@5
    q_focused    42      3   0.9280 0.9429 0.8461
    q_focused    43      2   0.9276 0.9419 0.8466
    q_focused    44      2   0.9273 0.9416 0.8446
q_focused_cat    42      3   0.9267 0.9409 0.8478
q_focused_cat    43      3   0.9284 0.9436 0.8495
q_focused_cat    44      2   0.9285 0.9414 0.8495
        q_raw    42      3   0.9277 0.9409 0.8470
        q_raw    43      2   0.9269 0.9422 0.8442
        q_raw    44      2   0.9287 0.9415 0.8459
    q_raw_cat    42      3   0.9294 0.9449 0.8489
    q_raw_cat    43      3   0.9282 0.9432 0.8489
    q_raw_cat    44      3   0.9294 0.9451 0.8491

mean ± std over 3 seeds:
               ndcg@10             mrr             p@5        
                 mean     std    mean     std    mean     std
qin                                                          
q_focused      0.9276  0.0004  0.9421  0.0007  0.8458  0.0011
q_focused_cat  0.9279  0.0010  0.9420  0.0014  0.8489 